# Multi-model comparison

Compare pipeline outputs across models: circuit evidence, steering success, and probe performance.
Run from repo root with `PYTHONPATH` set. Figures are saved to `outputs/synthesis/`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd() if (Path.cwd() / "pipeline").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline.config import OUTPUTS_ROOT, get_model_output_dir, get_circuit_dir, get_steering_dir, get_probes_dir

# Discover model_ids: dirs under outputs/ that have circuit_evidence_classification.csv
model_ids = []
for d in OUTPUTS_ROOT.iterdir():
    if d.is_dir() and (d / "02_circuit" / "circuit_evidence_classification.csv").exists():
        model_ids.append(d.name)
model_ids = sorted(model_ids)

syn_dir = OUTPUTS_ROOT / "synthesis"
if (syn_dir / "synthesis_metrics.csv").exists():
    metrics = pd.read_csv(syn_dir / "synthesis_metrics.csv")
    model_ids = sorted(metrics["model_id"].astype(str).tolist())
else:
    metrics = None

if not model_ids:
    print("No model outputs found. Run pipeline for at least one model and optionally run_synthesis(model_ids=[...]).")
else:
    print(f"Models to compare: {model_ids}")
    if metrics is not None:
        print(metrics.to_string())

## Build metrics table if needed

If synthesis_metrics.csv is missing, build it from each model's CSVs.

In [ ]:
def _collect_metrics(mid):
    row = {"model_id": mid}
    circuit_csv = get_circuit_dir(mid) / "circuit_evidence_classification.csv"
    if circuit_csv.exists():
        df = pd.read_csv(circuit_csv)
        sb = df[df["method"] == "single_best"]
        tk = df[df["method"] == "topk_fusion"]
        if not sb.empty:
            row["circuit_single_best_accuracy"] = float(sb["accuracy"].iloc[0])
            v = sb["roc_auc"].iloc[0]
            row["circuit_single_best_roc_auc"] = float(v) if pd.notna(v) else None
        if not tk.empty:
            row["circuit_topk_accuracy"] = float(tk["accuracy"].iloc[0])
            v = tk["roc_auc"].iloc[0]
            row["circuit_topk_roc_auc"] = float(v) if pd.notna(v) else None
    steer_csv = get_steering_dir(mid) / "steering_benchmark.csv"
    if steer_csv.exists():
        df = pd.read_csv(steer_csv)
        for _, r in df.iterrows():
            m = r.get("method", "")
            if m == "appraisal_steer":
                row["appraisal_steer_success_rate"] = float(r.get("success_rate", 0))
                row["appraisal_steer_mean_delta_logit"] = float(r.get("mean_delta_target_logit", 0))
            elif m == "emotion_steer":
                row["emotion_steer_success_rate"] = float(r.get("success_rate", 0))
                row["emotion_steer_mean_delta_logit"] = float(r.get("mean_delta_target_logit", 0))
    probe_csv = get_probes_dir(mid) / "binary_ova_probes" / "probe_summary.csv"
    if probe_csv.exists():
        ps = pd.read_csv(probe_csv)
        if "test_roc_auc" in ps.columns:
            row["mean_probe_test_roc_auc"] = float(ps["test_roc_auc"].mean())
    return row

if metrics is None and model_ids:
    metrics = pd.DataFrame([_collect_metrics(mid) for mid in model_ids])

def _family(mid):
    for prefix in ["Llama", "Gemma", "Phi", "Mistral", "OLMo"]:
        if str(mid).startswith(prefix):
            return prefix
    return "Other"

if metrics is not None and not metrics.empty:
    metrics["model_family"] = metrics["model_id"].astype(str).map(_family)
    display(metrics)

## Circuit evidence: single-best vs top-k accuracy by model

In [ ]:
if metrics is not None and "circuit_single_best_accuracy" in metrics.columns:
    fig, ax = plt.subplots(figsize=(max(8, len(model_ids) * 0.8), 5))
    x = np.arange(len(model_ids))
    w = 0.35
    sb = metrics["circuit_single_best_accuracy"].fillna(0)
    tk = metrics["circuit_topk_accuracy"].fillna(0)
    ax.bar(x - w/2, sb, w, label="Single-best", color="steelblue")
    ax.bar(x + w/2, tk, w, label="Top-k fusion", color="darkorange")
    ax.set_xticks(x)
    ax.set_xticklabels(model_ids, rotation=45, ha="right")
    ax.set_ylabel("Accuracy")
    ax.set_title("Circuit evidence: classification accuracy by model")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    syn_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(syn_dir / "comparison_circuit_accuracy.png", bbox_inches="tight", dpi=120)
    plt.show()
else:
    print("No circuit metrics to plot.")

## Steering: appraisal vs emotion success rate by model

In [ ]:
if metrics is not None and "emotion_steer_success_rate" in metrics.columns:
    fig, ax = plt.subplots(figsize=(max(8, len(model_ids) * 0.8), 5))
    x = np.arange(len(model_ids))
    w = 0.35
    app = metrics["appraisal_steer_success_rate"].fillna(0)
    emo = metrics["emotion_steer_success_rate"].fillna(0)
    ax.bar(x - w/2, app, w, label="Appraisal steer", color="seagreen")
    ax.bar(x + w/2, emo, w, label="Emotion steer", color="coral")
    ax.set_xticks(x)
    ax.set_xticklabels(model_ids, rotation=45, ha="right")
    ax.set_ylabel("Success rate")
    ax.set_title("Steering (anger→joy): success rate by model")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    fig.savefig(syn_dir / "comparison_steering_success.png", bbox_inches="tight", dpi=120)
    plt.show()
else:
    print("No steering metrics to plot.")

## Mean probe test ROC-AUC by model

In [ ]:
if metrics is not None and "mean_probe_test_roc_auc" in metrics.columns and metrics["mean_probe_test_roc_auc"].notna().any():
    fig, ax = plt.subplots(figsize=(max(8, len(model_ids) * 0.8), 5))
    x = np.arange(len(model_ids))
    roc = metrics["mean_probe_test_roc_auc"].fillna(0)
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_ids)))
    ax.bar(x, roc, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(model_ids, rotation=45, ha="right")
    ax.set_ylabel("Mean test ROC-AUC")
    ax.set_title("Probe decoding strength by model")
    ax.axhline(0.8, color="gray", linestyle="--", alpha=0.7, label="0.8 threshold")
    ax.set_ylim(0.5, 1.02)
    plt.tight_layout()
    fig.savefig(syn_dir / "comparison_probe_roc_auc.png", bbox_inches="tight", dpi=120)
    plt.show()
else:
    print("No probe ROC-AUC metrics to plot.")

## Summary: which model families achieve stronger results?

Short narrative from the metrics above.

In [ ]:
if metrics is not None and not metrics.empty:
    lines = []
    if "circuit_topk_accuracy" in metrics.columns and metrics["circuit_topk_accuracy"].notna().any():
        best = metrics.loc[metrics["circuit_topk_accuracy"].idxmax()]
        lines.append(f"**Best circuit (top-k) accuracy:** {best['model_id']} ({best['circuit_topk_accuracy']:.3f})")
    if "emotion_steer_success_rate" in metrics.columns and metrics["emotion_steer_success_rate"].notna().any():
        best = metrics.loc[metrics["emotion_steer_success_rate"].idxmax()]
        lines.append(f"**Best emotion steering success:** {best['model_id']} ({best['emotion_steer_success_rate']:.3f})")
    if "mean_probe_test_roc_auc" in metrics.columns and metrics["mean_probe_test_roc_auc"].notna().any():
        best = metrics.loc[metrics["mean_probe_test_roc_auc"].idxmax()]
        lines.append(f"**Best mean probe ROC-AUC:** {best['model_id']} ({best['mean_probe_test_roc_auc']:.3f})")
    if "model_family" in metrics.columns:
        by_family = metrics.groupby("model_family").agg({
            "circuit_topk_accuracy": "mean",
            "emotion_steer_success_rate": "mean",
        }).round(3)
        lines.append("")
        lines.append("**Averages by model family:**")
        lines.append(by_family.to_string())
    for L in lines:
        print(L)
else:
    print("No metrics to summarize.")